# Timepoint Terminal Supervised Embedding

This notebook implements a third protein-expression embedding workflow for the embryogenesis bundle.

It reuses the two-stage supervised pipeline in `feature_selection.py`, but treats individual time points from typed terminal cells as training samples. Cross-validation is lineage-cell based, so all time points from the same lineage stay in the same fold.

Before running the notebook, select the `dev` conda kernel.

## 1. Import Dependencies

The workflow uses the existing supervised embedding utilities from `feature_selection.py`, plus `anndata` for the per-timepoint protein matrix and plotting helpers for grouped-CV diagnostics.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import json
import sys

import anndata as ad
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix

mpl.rcParams["figure.dpi"] = 300

cwd = Path.cwd().resolve()
if (cwd / "data").exists() and (cwd / "expression_embedding").exists():
    ROOT_DIR = cwd
    BUNDLE_DIR = ROOT_DIR / "expression_embedding"
else:
    BUNDLE_DIR = cwd
    ROOT_DIR = BUNDLE_DIR.parents[0]

if str(BUNDLE_DIR) not in sys.path:
    sys.path.insert(0, str(BUNDLE_DIR))

from feature_selection import (
    build_cv_folds,
    build_selector_fold_cache,
    cross_validate_features,
    cross_validate_focused,
    eval_prediction_metrics,
    train_focused,
    train_one_pass,
 )

print(f"Root directory  : {ROOT_DIR}")
print(f"Bundle directory: {BUNDLE_DIR}")

Root directory  : /Users/bingranshen/code/embryogenesis
Bundle directory: /Users/bingranshen/code/embryogenesis/expression_embedding


## 2. Set Up Configuration and Sample Inputs

These settings keep the workflow comparable to the existing supervised notebook while adding lineage-grouped splits for timepoint data. Set `MAX_LINEAGE_GROUPS` to a small integer when you want a quick smoke test.

In [ ]:
RESULTS_DIR = BUNDLE_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

AGGREGATED_SCANPY_PATH = ROOT_DIR / "data" / "protein" / "aggregated_scanpy.h5ad"
CELL_LINEAGE_PATH = ROOT_DIR / "data" / "cell_lineage.json"
CELL_TYPE_PATH = ROOT_DIR / "data" / "2023-06-29_entropy_cell_key_V2.csv"

PHASE1_PARAM_GRID = {
    "l1_lambda": [0.002, 0.004],
    "hidden_dims": [(128, 64), (64, 32)],
    "dropout": [0.2, 0.3],
}
PHASE1_SCORE_WEIGHTS = {
    "val_acc": 0.60,
    "selection_stability": 0.40,
    "overfit_gap": 0.10,
}
PHASE1_STABILITY_WEIGHTS = {
    "jaccard": 0.50,
    "topk_frequency": 0.35,
    "spearman": 0.15,
}
PHASE2_PARAM_GRID = {
    "n_select": [10, 15, 20],
    "hidden_dims": [(32,), (64, 32)],
    "dropout": [0.1, 0.2],
    "dist_lambda": [0.0, 0.1],
}

N_SPLITS = 5
PHASE1_STABILITY_K = 20
PHASE1_CV_EPOCHS = 150
PHASE2_SELECTOR_EPOCHS = 120
PHASE2_CV_EPOCHS = 220
FINAL_PHASE1_EPOCHS = 300
FINAL_PHASE2_EPOCHS = 400
PHASE1_EARLY_STOPPING_PATIENCE = 20
PHASE2_SELECTOR_EARLY_STOPPING_PATIENCE = 15
PHASE2_FOCUSED_EARLY_STOPPING_PATIENCE = 20
FINAL_PHASE1_EARLY_STOPPING_PATIENCE = 25
FINAL_PHASE2_EARLY_STOPPING_PATIENCE = 25
CV_VERBOSE = True
TIME_BIN_COUNT = 12
MAX_LINEAGE_GROUPS = None  # Set to an integer for a reduced smoke test.
SEED = 42

for required_path in [AGGREGATED_SCANPY_PATH, CELL_LINEAGE_PATH, CELL_TYPE_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)

device = (
    torch.device("mps")
    if torch.backends.mps.is_available()
    else torch.device("cuda")
    if torch.cuda.is_available()
    else torch.device("cpu")
)

print(f"Device: {device}")
print(f"Results directory: {RESULTS_DIR}")
print(f"MAX_LINEAGE_GROUPS: {MAX_LINEAGE_GROUPS}")
print(f"CV verbose logging: {CV_VERBOSE}")

Device: mps
Results directory: /Users/bingranshen/code/embryogenesis/expression_embedding/results
MAX_LINEAGE_GROUPS: None


## 3. Define Core Data Structures

The workflow runs two dataset variants: one excludes programmed-death cells, and the other keeps them as an explicit supervised class for comparison.

In [3]:
@dataclass(frozen=True)
class VariantConfig:
    name: str
    include_programmed_death: bool
    file_stem: str


VARIANTS = [
    VariantConfig(
        name="exclude_programmed_death",
        include_programmed_death=False,
        file_stem="timepoint_terminal_exclude_pd",
    ),
    VariantConfig(
        name="include_programmed_death",
        include_programmed_death=True,
        file_stem="timepoint_terminal_include_pd",
    ),
]

pd.DataFrame([variant.__dict__ for variant in VARIANTS])

,name,include_programmed_death,file_stem
0,exclude_programmed_death,False,timepoint_terminal_exclude_pd
1,include_programmed_death,True,timepoint_terminal_include_pd


## 4. Implement the Main Functions

These helpers keep the workflow modular: dataset construction, grouped evaluation, final model fitting, export, and visualization all live in one place so the execution section stays compact.

In [4]:
def map_names(did: str) -> str:
    if did == "P4a":
        return "Z3"
    if did == "P4p":
        return "Z2"
    if did == "P0a":
        return "AB"
    return did


def collect_terminal_cell_types(cell_lineage_path: Path, cell_type_path: Path):
    with cell_lineage_path.open("r", encoding="utf-8") as handle:
        lineage_data = json.load(handle)

    terminal_nodes = []

    def dfs(node):
        lookup_name = map_names(node["did"])
        children = node.get("children", [])
        if len(children) == 0:
            terminal_nodes.append(lookup_name)
            return
        for child in children:
            dfs(child)

    dfs(lineage_data)
    terminal_nodes = sorted(set(terminal_nodes))

    cell_type_df = pd.read_csv(cell_type_path)
    lineage_to_type = {}
    for node in terminal_nodes:
        cur_types = (
            cell_type_df.loc[cell_type_df["wormweb.lineage"] == node, "wormweb.type"]
            .dropna()
            .unique()
        )
        if len(cur_types) == 0:
            lineage_to_type[node] = "programmed_death"
        else:
            lineage_to_type[node] = cur_types[0]

    return terminal_nodes, lineage_to_type


def load_timepoint_matrix(h5ad_path: Path):
    adata = ad.read_h5ad(h5ad_path)
    expr_df = adata.to_df().fillna(0.0)
    obs_df = adata.obs.copy()
    obs_df.index.name = "obs_id"
    obs_df["obs_id"] = obs_df.index.astype(str)
    obs_df["Time"] = obs_df["Time"].astype(int)
    obs_df["lineage_name"] = obs_df["Cell-name"].map(map_names)
    return expr_df, obs_df


def zscore_feature_matrix(expr_df: pd.DataFrame):
    expr_df = expr_df.loc[:, ~(expr_df == 0).all(axis=0)].copy()
    feature_mean = expr_df.mean(axis=0)
    feature_std = expr_df.std(axis=0).replace(0.0, 1.0)
    zscored_df = ((expr_df - feature_mean) / feature_std).fillna(0.0)
    return zscored_df, feature_mean, feature_std


def build_terminal_dataset(
    zscored_df: pd.DataFrame,
    obs_df: pd.DataFrame,
    lineage_to_type: dict,
    include_programmed_death: bool = False,
    max_lineage_groups: int | None = None,
):
    metadata = obs_df.reset_index(drop=True).copy()
    metadata["obs_id"] = metadata["obs_id"].astype(str)
    metadata["cell_type"] = metadata["lineage_name"].map(lineage_to_type)
    metadata = metadata[metadata["cell_type"].notna()].copy()

    if not include_programmed_death:
        metadata = metadata[metadata["cell_type"] != "programmed_death"].copy()

    if max_lineage_groups is not None:
        lineage_counts = metadata["lineage_name"].value_counts().sort_values(ascending=False)
        keep_lineages = lineage_counts.index[:max_lineage_groups]
        metadata = metadata[metadata["lineage_name"].isin(keep_lineages)].copy()

    if metadata.empty:
        raise ValueError("No terminal timepoint samples remain after filtering.")

    class_counts = metadata["cell_type"].value_counts()
    class_names = class_counts.index.tolist()
    if include_programmed_death and "programmed_death" in class_names:
        class_names = [name for name in class_names if name != "programmed_death"] + ["programmed_death"]

    class_to_idx = {class_name: idx for idx, class_name in enumerate(class_names)}
    metadata = metadata.sort_values(["lineage_name", "Time", "obs_id"]).copy()
    metadata["label_idx"] = metadata["cell_type"].map(class_to_idx)

    y = np.eye(len(class_names), dtype=np.float32)[metadata["label_idx"].to_numpy()]
    X = zscored_df.loc[metadata["obs_id"]].to_numpy(dtype=np.float32)
    groups = metadata["lineage_name"].to_numpy()
    sample_weights = np.ones(len(metadata), dtype=np.float32)
    terminal_mask = np.ones(len(metadata), dtype=bool)
    feature_names = zscored_df.columns.to_numpy()

    metadata.index = metadata["obs_id"]
    metadata.index.name = "obs_id"
    metadata = metadata.loc[:, ["obs_id", "lineage_name", "Time", "cell_type", "label_idx"]].copy()

    return {
        "X": X,
        "y": y,
        "groups": groups,
        "sample_weights": sample_weights,
        "terminal_mask": terminal_mask,
        "feature_names": feature_names,
        "class_names": class_names,
        "metadata": metadata,
        "n_groups": int(pd.Series(groups).nunique()),
    }


def summarize_cv_results(cv_results):
    summary_rows = []
    for result in cv_results:
        summary_rows.append({
            **result["config"],
            **result["summary"],
            "score": result["score"],
        })
    return pd.DataFrame(summary_rows).sort_values("score", ascending=False)

In [5]:
def run_grouped_holdout_predictions(
    dataset: dict,
    best_selector_config: dict,
    best_phase2_config: dict,
    n_splits: int,
    selector_epochs: int,
    focused_epochs: int,
    seed: int,
    device: torch.device,
):
    X = dataset["X"]
    y = dataset["y"]
    groups = dataset["groups"]
    sample_weights = dataset["sample_weights"]
    feature_names = dataset["feature_names"]
    class_names = dataset["class_names"]
    metadata = dataset["metadata"].copy()

    fold_assignments = np.full(len(X), -1, dtype=int)
    probs = np.zeros((len(X), y.shape[1]), dtype=np.float32)
    selected_feature_names_per_fold = []

    folds = build_cv_folds(X, y, n_splits=n_splits, seed=seed, groups=groups)
    for fold_idx, (train_idx, val_idx) in enumerate(folds):
        _, top_idx, _, _, _ = train_one_pass(
            X[train_idx],
            y[train_idx],
            sample_weights[train_idx],
            hidden_dims=best_selector_config["hidden_dims"],
            l1_lambda=best_selector_config["l1_lambda"],
            dropout=best_selector_config.get("dropout", 0.3),
            n_select=best_phase2_config["n_select"],
            n_epochs=selector_epochs,
            seed=seed + fold_idx,
            device=str(device),
        )

        selected_feature_names_per_fold.append(feature_names[top_idx].tolist())
        X_tr_sel = X[train_idx][:, top_idx].astype(np.float32)
        X_val_sel = X[val_idx][:, top_idx].astype(np.float32)

        model_final, _, _ = train_focused(
            X_tr_sel,
            y[train_idx],
            sample_weights[train_idx],
            hidden_dims=best_phase2_config["hidden_dims"],
            dropout=best_phase2_config.get("dropout", 0.2),
            n_epochs=focused_epochs,
            batch_size=64,
            seed=seed + fold_idx,
            device=str(device),
            dist_lambda=best_phase2_config.get("dist_lambda", 0.0),
        )

        with torch.no_grad():
            logits_val, _ = model_final(torch.FloatTensor(X_val_sel).to(device))
        probs[val_idx] = torch.softmax(logits_val, dim=-1).cpu().numpy()
        fold_assignments[val_idx] = fold_idx

    oof_df = metadata.copy()
    oof_df["fold"] = fold_assignments
    oof_df["true_idx"] = y.argmax(axis=1)
    oof_df["pred_idx"] = probs.argmax(axis=1)
    oof_df["true_type"] = [class_names[idx] for idx in oof_df["true_idx"]]
    oof_df["pred_type"] = [class_names[idx] for idx in oof_df["pred_idx"]]
    oof_df["correct"] = oof_df["true_idx"] == oof_df["pred_idx"]
    oof_df["confidence"] = probs.max(axis=1)
    return oof_df, probs, selected_feature_names_per_fold


def train_final_embedding_models(
    dataset: dict,
    best_selector_config: dict,
    best_phase2_config: dict,
    final_phase1_epochs: int,
    final_phase2_epochs: int,
    seed: int,
    device: torch.device,
):
    X = dataset["X"]
    y = dataset["y"]
    sample_weights = dataset["sample_weights"]

    model_sparse, top_idx, gate_values, _, phase1_losses = train_one_pass(
        X,
        y,
        sample_weights,
        hidden_dims=best_selector_config["hidden_dims"],
        l1_lambda=best_selector_config["l1_lambda"],
        dropout=best_selector_config.get("dropout", 0.3),
        n_select=best_phase2_config["n_select"],
        n_epochs=final_phase1_epochs,
        seed=seed,
        device=str(device),
    )

    X_sel = X[:, top_idx].astype(np.float32)
    model_final, embeddings, phase2_losses = train_focused(
        X_sel,
        y,
        sample_weights,
        hidden_dims=best_phase2_config["hidden_dims"],
        dropout=best_phase2_config.get("dropout", 0.2),
        n_epochs=final_phase2_epochs,
        batch_size=64,
        seed=seed,
        device=str(device),
        dist_lambda=best_phase2_config.get("dist_lambda", 0.0),
    )

    X_sel_t = torch.FloatTensor(X_sel)
    y_t = torch.FloatTensor(y)
    final_metrics = eval_prediction_metrics(
        model_final,
        X_sel_t,
        y_t,
        dataset["terminal_mask"],
        device,
        sample_weights=sample_weights,
    )

    emb_cols = [f"emb_{idx}" for idx in range(embeddings.shape[1])]
    embeddings_df = pd.DataFrame(embeddings, index=dataset["metadata"].index, columns=emb_cols)
    embeddings_df.index.name = "obs_id"

    metadata_df = dataset["metadata"].copy()
    metadata_df["variant"] = dataset["variant_name"]

    return {
        "selector_model": model_sparse,
        "focused_model": model_final,
        "top_idx": top_idx,
        "gate_values": gate_values,
        "selected_features": dataset["feature_names"][top_idx].tolist(),
        "phase1_losses": phase1_losses,
        "phase2_losses": phase2_losses,
        "final_metrics": final_metrics,
        "embeddings_df": embeddings_df,
        "metadata_df": metadata_df,
        "X_sel": X_sel,
    }


def save_variant_outputs(
    variant: VariantConfig,
    trained_bundle: dict,
    phase1_summary_df: pd.DataFrame,
    phase2_summary_df: pd.DataFrame,
    oof_df: pd.DataFrame,
    results_dir: Path,
):
    embed_dim = trained_bundle["embeddings_df"].shape[1]
    saved_paths = {
        "phase1_summary": results_dir / f"{variant.file_stem}_phase1_cv_summary.csv",
        "phase2_summary": results_dir / f"{variant.file_stem}_phase2_cv_summary.csv",
        "embeddings": results_dir / f"{variant.file_stem}_embeddings_{embed_dim}d.csv",
        "metadata": results_dir / f"{variant.file_stem}_embedding_metadata.csv",
        "selected_features": results_dir / f"{variant.file_stem}_selected_proteins.tsv",
        "oof_predictions": results_dir / f"{variant.file_stem}_grouped_oof_predictions.csv",
    }

    phase1_summary_df.to_csv(saved_paths["phase1_summary"], index=False)
    phase2_summary_df.to_csv(saved_paths["phase2_summary"], index=False)
    trained_bundle["embeddings_df"].to_csv(saved_paths["embeddings"])
    trained_bundle["metadata_df"].to_csv(saved_paths["metadata"])
    pd.Series(trained_bundle["selected_features"]).to_csv(
        saved_paths["selected_features"],
        sep="\t",
        index=False,
        header=False,
    )
    oof_df.to_csv(saved_paths["oof_predictions"], index=True)
    return saved_paths


def compute_time_bin_accuracy(oof_df: pd.DataFrame, time_bin_count: int = 12):
    n_bins = min(time_bin_count, oof_df["Time"].nunique())
    time_bins = pd.qcut(oof_df["Time"], q=n_bins, duplicates="drop")
    time_df = oof_df.assign(time_bin=time_bins.astype(str))
    summary = (
        time_df.groupby("time_bin")
        .agg(
            mean_time=("Time", "mean"),
            accuracy=("correct", "mean"),
            n=("correct", "size"),
        )
        .reset_index()
        .sort_values("mean_time")
    )
    return summary


def plot_variant_diagnostics(bundle: dict):
    dataset = bundle["dataset"]
    class_names = dataset["class_names"]
    oof_df = bundle["oof_df"]
    embeddings_df = bundle["embeddings_df"]
    metadata_df = bundle["metadata_df"]
    feature_freq = bundle["phase1_results"][0]["stability"]["feature_frequency"]
    feature_names = dataset["feature_names"]
    top_freq_idx = np.argsort(feature_freq)[::-1][: min(15, len(feature_freq))]

    fig, axes = plt.subplots(2, 2, figsize=(18, 14))

    conf = confusion_matrix(
        oof_df["true_idx"],
        oof_df["pred_idx"],
        labels=np.arange(len(class_names)),
    )
    annot = len(class_names) <= 12
    sns.heatmap(
        conf,
        annot=annot,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
        ax=axes[0, 0],
        cbar=False,
    )
    axes[0, 0].set_title(f"{bundle['variant'].name}: grouped OOF confusion matrix")
    axes[0, 0].set_xlabel("Predicted")
    axes[0, 0].set_ylabel("True")
    axes[0, 0].tick_params(axis="x", rotation=45)
    axes[0, 0].tick_params(axis="y", rotation=0)

    time_accuracy_df = compute_time_bin_accuracy(oof_df, time_bin_count=TIME_BIN_COUNT)
    axes[0, 1].plot(time_accuracy_df["mean_time"], time_accuracy_df["accuracy"], marker="o")
    axes[0, 1].set_title(f"{bundle['variant'].name}: accuracy by time bin")
    axes[0, 1].set_xlabel("Mean time")
    axes[0, 1].set_ylabel("Grouped OOF accuracy")
    axes[0, 1].set_ylim(0.0, 1.05)

    axes[1, 0].bar(
        range(len(top_freq_idx)),
        feature_freq[top_freq_idx],
        color="steelblue",
        edgecolor="white",
    )
    axes[1, 0].set_xticks(range(len(top_freq_idx)))
    axes[1, 0].set_xticklabels(feature_names[top_freq_idx], rotation=45, ha="right", fontsize=8)
    axes[1, 0].set_ylabel("Fold frequency")
    axes[1, 0].set_title(f"{bundle['variant'].name}: Phase-1 feature stability")

    pca = PCA(n_components=2)
    coords = pca.fit_transform(embeddings_df.to_numpy())
    scatter_df = metadata_df.copy()
    scatter_df["pc1"] = coords[:, 0]
    scatter_df["pc2"] = coords[:, 1]
    sns.scatterplot(
        data=scatter_df,
        x="pc1",
        y="pc2",
        hue="cell_type",
        s=10,
        alpha=0.45,
        linewidth=0,
        legend=False,
        ax=axes[1, 1],
    )
    axes[1, 1].set_title(f"{bundle['variant'].name}: PCA of final embeddings")
    axes[1, 1].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
    axes[1, 1].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")

    plt.tight_layout()
    plt.show()


def run_variant_workflow(
    variant: VariantConfig,
    zscored_df: pd.DataFrame,
    obs_df: pd.DataFrame,
    lineage_to_type: dict,
    device: torch.device,
):
    dataset = build_terminal_dataset(
        zscored_df,
        obs_df,
        lineage_to_type,
        include_programmed_death=variant.include_programmed_death,
        max_lineage_groups=MAX_LINEAGE_GROUPS,
    )
    dataset["variant_name"] = variant.name

    print(
        f"\n[{variant.name}] samples={dataset['X'].shape[0]}, "
        f"features={dataset['X'].shape[1]}, groups={dataset['n_groups']}, "
        f"classes={len(dataset['class_names'])}"
    )
    print(pd.Series(dataset["metadata"]["cell_type"]).value_counts().head(10))

    phase1_results, best_selector_config = cross_validate_features(
        dataset["X"],
        dataset["y"],
        dataset["sample_weights"],
        dataset["terminal_mask"],
        param_grid=PHASE1_PARAM_GRID,
        groups=dataset["groups"],
        n_splits=N_SPLITS,
        n_select=PHASE1_STABILITY_K,
        n_epochs=PHASE1_CV_EPOCHS,
        seed=SEED,
        device=str(device),
        score_weights=PHASE1_SCORE_WEIGHTS,
        stability_metric_weights=PHASE1_STABILITY_WEIGHTS,
    )
    phase1_summary_df = summarize_cv_results(phase1_results)

    phase2_results, best_phase2_config = cross_validate_focused(
        dataset["X"],
        dataset["y"],
        dataset["sample_weights"],
        dataset["terminal_mask"],
        selector_config=best_selector_config,
        param_grid=PHASE2_PARAM_GRID,
        groups=dataset["groups"],
        n_splits=N_SPLITS,
        selector_epochs=PHASE2_SELECTOR_EPOCHS,
        focused_epochs=PHASE2_CV_EPOCHS,
        selector_batch_size=128,
        focused_batch_size=64,
        seed=SEED,
        device=str(device),
    )
    phase2_summary_df = summarize_cv_results(phase2_results)

    oof_df, oof_probs, selected_feature_names_per_fold = run_grouped_holdout_predictions(
        dataset,
        best_selector_config,
        best_phase2_config,
        n_splits=N_SPLITS,
        selector_epochs=PHASE2_SELECTOR_EPOCHS,
        focused_epochs=PHASE2_CV_EPOCHS,
        seed=SEED,
        device=device,
    )

    trained_bundle = train_final_embedding_models(
        dataset,
        best_selector_config,
        best_phase2_config,
        final_phase1_epochs=FINAL_PHASE1_EPOCHS,
        final_phase2_epochs=FINAL_PHASE2_EPOCHS,
        seed=SEED,
        device=device,
    )
    saved_paths = save_variant_outputs(
        variant,
        trained_bundle,
        phase1_summary_df,
        phase2_summary_df,
        oof_df,
        RESULTS_DIR,
    )

    return {
        "variant": variant,
        "dataset": dataset,
        "phase1_results": phase1_results,
        "phase1_summary_df": phase1_summary_df,
        "best_selector_config": best_selector_config,
        "phase2_results": phase2_results,
        "phase2_summary_df": phase2_summary_df,
        "best_phase2_config": best_phase2_config,
        "oof_df": oof_df,
        "oof_probs": oof_probs,
        "selected_feature_names_per_fold": selected_feature_names_per_fold,
        "saved_paths": saved_paths,
        **trained_bundle,
    }

In [ ]:
def compute_prediction_summary(y_true: np.ndarray, probs: np.ndarray, sample_weights: np.ndarray):
    clipped_probs = np.clip(probs, 1e-8, 1.0)
    weights = sample_weights.astype(np.float64)
    if weights.sum() <= 0:
        weights = np.ones_like(weights, dtype=np.float64)
    return {
        "argmax_accuracy": float(
            np.average(
                (probs.argmax(axis=1) == y_true.argmax(axis=1)).astype(np.float32),
                weights=weights,
            )
        ),
        "expected_target_probability": float(
            np.average(np.sum(y_true * probs, axis=1), weights=weights)
        ),
        "soft_cross_entropy": float(
            np.average(-np.sum(y_true * np.log(clipped_probs), axis=1), weights=weights)
        ),
    }


def run_grouped_holdout_predictions(
    dataset: dict,
    best_phase2_config: dict,
    selector_cache: list[dict],
    focused_epochs: int,
    seed: int,
    device: torch.device,
    focused_early_stopping_patience: int | None = None,
    verbose: bool = False,
):
    X = dataset["X"]
    y = dataset["y"]
    sample_weights = dataset["sample_weights"]
    feature_names = dataset["feature_names"]
    class_names = dataset["class_names"]
    metadata = dataset["metadata"].copy()

    fold_assignments = np.full(len(X), -1, dtype=int)
    probs = np.zeros((len(X), y.shape[1]), dtype=np.float32)
    selected_feature_names_per_fold = []

    for fold_number, fold_info in enumerate(selector_cache, start=1):
        train_idx = fold_info["train_idx"]
        val_idx = fold_info["val_idx"]
        top_idx = fold_info["feature_ranking"][: best_phase2_config["n_select"]]

        selected_feature_names_per_fold.append(feature_names[top_idx].tolist())
        X_tr_sel = X[train_idx][:, top_idx].astype(np.float32)
        X_val_sel = X[val_idx][:, top_idx].astype(np.float32)

        model_final, _, losses = train_focused(
            X_tr_sel,
            y[train_idx],
            sample_weights[train_idx],
            hidden_dims=best_phase2_config["hidden_dims"],
            dropout=best_phase2_config.get("dropout", 0.2),
            n_epochs=focused_epochs,
            batch_size=64,
            seed=seed + fold_number - 1,
            device=str(device),
            dist_lambda=best_phase2_config.get("dist_lambda", 0.0),
            early_stopping_patience=focused_early_stopping_patience,
        )

        with torch.no_grad():
            logits_val, _ = model_final(torch.FloatTensor(X_val_sel).to(device))
        val_probs = torch.softmax(logits_val, dim=-1).cpu().numpy()
        probs[val_idx] = val_probs
        fold_assignments[val_idx] = fold_info["fold"]

        fold_metrics = compute_prediction_summary(
            y[val_idx],
            val_probs,
            sample_weights[val_idx],
        )
        if verbose:
            print(
                f"[{dataset['variant_name']} oof] fold {fold_number}/{len(selector_cache)} "
                f"argmax_acc={fold_metrics['argmax_accuracy']:.4f} "
                f"expected_target_prob={fold_metrics['expected_target_probability']:.4f} "
                f"soft_ce={fold_metrics['soft_cross_entropy']:.4f} "
                f"selector_epochs={fold_info.get('selector_epochs_ran')} "
                f"focused_epochs={len(losses)}"
            )

    oof_metrics = compute_prediction_summary(y, probs, sample_weights)
    if verbose:
        print(
            f"[{dataset['variant_name']} oof] overall "
            f"argmax_acc={oof_metrics['argmax_accuracy']:.4f} "
            f"expected_target_prob={oof_metrics['expected_target_probability']:.4f} "
            f"soft_ce={oof_metrics['soft_cross_entropy']:.4f}"
        )

    oof_df = metadata.copy()
    oof_df["fold"] = fold_assignments
    oof_df["true_idx"] = y.argmax(axis=1)
    oof_df["pred_idx"] = probs.argmax(axis=1)
    oof_df["true_type"] = [class_names[idx] for idx in oof_df["true_idx"]]
    oof_df["pred_type"] = [class_names[idx] for idx in oof_df["pred_idx"]]
    oof_df["correct"] = oof_df["true_idx"] == oof_df["pred_idx"]
    oof_df["confidence"] = probs.max(axis=1)
    return oof_df, probs, selected_feature_names_per_fold, oof_metrics


def train_final_embedding_models(
    dataset: dict,
    best_selector_config: dict,
    best_phase2_config: dict,
    final_phase1_epochs: int,
    final_phase2_epochs: int,
    seed: int,
    device: torch.device,
    phase1_early_stopping_patience: int | None = None,
    phase2_early_stopping_patience: int | None = None,
    verbose: bool = False,
):
    X = dataset["X"]
    y = dataset["y"]
    sample_weights = dataset["sample_weights"]

    model_sparse, top_idx, gate_values, _, phase1_losses = train_one_pass(
        X,
        y,
        sample_weights,
        hidden_dims=best_selector_config["hidden_dims"],
        l1_lambda=best_selector_config["l1_lambda"],
        dropout=best_selector_config.get("dropout", 0.3),
        n_select=best_phase2_config["n_select"],
        n_epochs=final_phase1_epochs,
        seed=seed,
        device=str(device),
        early_stopping_patience=phase1_early_stopping_patience,
    )

    X_sel = X[:, top_idx].astype(np.float32)
    model_final, embeddings, phase2_losses = train_focused(
        X_sel,
        y,
        sample_weights,
        hidden_dims=best_phase2_config["hidden_dims"],
        dropout=best_phase2_config.get("dropout", 0.2),
        n_epochs=final_phase2_epochs,
        batch_size=64,
        seed=seed,
        device=str(device),
        dist_lambda=best_phase2_config.get("dist_lambda", 0.0),
        early_stopping_patience=phase2_early_stopping_patience,
    )

    X_sel_t = torch.FloatTensor(X_sel)
    y_t = torch.FloatTensor(y)
    final_metrics = eval_prediction_metrics(
        model_final,
        X_sel_t,
        y_t,
        dataset["terminal_mask"],
        device,
        sample_weights=sample_weights,
    )

    if verbose:
        print(
            f"[{dataset['variant_name']} final] phase1_epochs={len(phase1_losses)} "
            f"phase2_epochs={len(phase2_losses)} "
            f"terminal_acc={final_metrics['argmax_accuracy']:.4f} "
            f"expected_target_prob={final_metrics['expected_target_probability']:.4f} "
            f"soft_ce={final_metrics['soft_cross_entropy']:.4f}"
        )

    emb_cols = [f"emb_{idx}" for idx in range(embeddings.shape[1])]
    embeddings_df = pd.DataFrame(embeddings, index=dataset["metadata"].index, columns=emb_cols)
    embeddings_df.index.name = "obs_id"

    metadata_df = dataset["metadata"].copy()
    metadata_df["variant"] = dataset["variant_name"]

    return {
        "selector_model": model_sparse,
        "focused_model": model_final,
        "top_idx": top_idx,
        "gate_values": gate_values,
        "selected_features": dataset["feature_names"][top_idx].tolist(),
        "phase1_losses": phase1_losses,
        "phase2_losses": phase2_losses,
        "phase1_epochs_ran": len(phase1_losses),
        "phase2_epochs_ran": len(phase2_losses),
        "final_metrics": final_metrics,
        "embeddings_df": embeddings_df,
        "metadata_df": metadata_df,
        "X_sel": X_sel,
    }


def run_variant_workflow(
    variant: VariantConfig,
    zscored_df: pd.DataFrame,
    obs_df: pd.DataFrame,
    lineage_to_type: dict,
    device: torch.device,
):
    dataset = build_terminal_dataset(
        zscored_df,
        obs_df,
        lineage_to_type,
        include_programmed_death=variant.include_programmed_death,
        max_lineage_groups=MAX_LINEAGE_GROUPS,
    )
    dataset["variant_name"] = variant.name

    print(
        f"\n[{variant.name}] samples={dataset['X'].shape[0]}, "
        f"features={dataset['X'].shape[1]}, groups={dataset['n_groups']}, "
        f"classes={len(dataset['class_names'])}"
    )
    print(pd.Series(dataset["metadata"]["cell_type"]).value_counts().head(10))

    phase1_results, best_selector_config = cross_validate_features(
        dataset["X"],
        dataset["y"],
        dataset["sample_weights"],
        dataset["terminal_mask"],
        param_grid=PHASE1_PARAM_GRID,
        groups=dataset["groups"],
        n_splits=N_SPLITS,
        n_select=PHASE1_STABILITY_K,
        n_epochs=PHASE1_CV_EPOCHS,
        seed=SEED,
        device=str(device),
        score_weights=PHASE1_SCORE_WEIGHTS,
        stability_metric_weights=PHASE1_STABILITY_WEIGHTS,
        early_stopping_patience=PHASE1_EARLY_STOPPING_PATIENCE,
        verbose=CV_VERBOSE,
        log_prefix=f"{variant.name} phase1",
    )
    phase1_summary_df = summarize_cv_results(phase1_results)
    print(
        f"[{variant.name}] best phase1 config={best_selector_config} "
        f"score={phase1_results[0]['score']:.4f}"
    )

    folds = build_cv_folds(
        dataset["X"],
        dataset["y"],
        n_splits=N_SPLITS,
        seed=SEED,
        groups=dataset["groups"],
    )
    selector_cache = build_selector_fold_cache(
        dataset["X"],
        dataset["y"],
        dataset["sample_weights"],
        best_selector_config,
        folds,
        selector_epochs=PHASE2_SELECTOR_EPOCHS,
        selector_batch_size=128,
        seed=SEED,
        device=str(device),
        early_stopping_patience=PHASE2_SELECTOR_EARLY_STOPPING_PATIENCE,
        verbose=CV_VERBOSE,
        log_prefix=f"{variant.name} selector",
    )

    phase2_results, best_phase2_config = cross_validate_focused(
        dataset["X"],
        dataset["y"],
        dataset["sample_weights"],
        dataset["terminal_mask"],
        selector_config=best_selector_config,
        param_grid=PHASE2_PARAM_GRID,
        groups=dataset["groups"],
        n_splits=N_SPLITS,
        selector_epochs=PHASE2_SELECTOR_EPOCHS,
        focused_epochs=PHASE2_CV_EPOCHS,
        selector_batch_size=128,
        focused_batch_size=64,
        seed=SEED,
        device=str(device),
        selector_cache=selector_cache,
        focused_early_stopping_patience=PHASE2_FOCUSED_EARLY_STOPPING_PATIENCE,
        verbose=CV_VERBOSE,
        log_prefix=f"{variant.name} phase2",
    )
    phase2_summary_df = summarize_cv_results(phase2_results)
    print(
        f"[{variant.name}] best phase2 config={best_phase2_config} "
        f"score={phase2_results[0]['score']:.4f}"
    )

    oof_df, oof_probs, selected_feature_names_per_fold, oof_metrics = run_grouped_holdout_predictions(
        dataset,
        best_phase2_config,
        selector_cache,
        focused_epochs=PHASE2_CV_EPOCHS,
        seed=SEED,
        device=device,
        focused_early_stopping_patience=PHASE2_FOCUSED_EARLY_STOPPING_PATIENCE,
        verbose=CV_VERBOSE,
    )

    trained_bundle = train_final_embedding_models(
        dataset,
        best_selector_config,
        best_phase2_config,
        final_phase1_epochs=FINAL_PHASE1_EPOCHS,
        final_phase2_epochs=FINAL_PHASE2_EPOCHS,
        seed=SEED,
        device=device,
        phase1_early_stopping_patience=FINAL_PHASE1_EARLY_STOPPING_PATIENCE,
        phase2_early_stopping_patience=FINAL_PHASE2_EARLY_STOPPING_PATIENCE,
        verbose=CV_VERBOSE,
    )
    saved_paths = save_variant_outputs(
        variant,
        trained_bundle,
        phase1_summary_df,
        phase2_summary_df,
        oof_df,
        RESULTS_DIR,
    )

    return {
        "variant": variant,
        "dataset": dataset,
        "phase1_results": phase1_results,
        "phase1_summary_df": phase1_summary_df,
        "best_selector_config": best_selector_config,
        "phase2_results": phase2_results,
        "phase2_summary_df": phase2_summary_df,
        "best_phase2_config": best_phase2_config,
        "selector_cache": selector_cache,
        "oof_df": oof_df,
        "oof_probs": oof_probs,
        "oof_metrics": oof_metrics,
        "selected_feature_names_per_fold": selected_feature_names_per_fold,
        "saved_paths": saved_paths,
        **trained_bundle,
    }

## 5. Run the Initial Workflow

The workflow loads the per-timepoint matrix once, constructs terminal-only datasets for both label variants, runs grouped Phase 1 and Phase 2 CV, fits final models, and writes bundle-local outputs.

In [6]:
terminal_nodes, lineage_to_type = collect_terminal_cell_types(CELL_LINEAGE_PATH, CELL_TYPE_PATH)
expr_df, obs_df = load_timepoint_matrix(AGGREGATED_SCANPY_PATH)
zscored_df, feature_mean, feature_std = zscore_feature_matrix(expr_df)

print(f"Terminal lineage count: {len(terminal_nodes)}")
print(f"Timepoint matrix shape: {expr_df.shape}")
print(f"Z-scored matrix shape: {zscored_df.shape}")

variant_results = {}
for variant in VARIANTS:
    variant_results[variant.name] = run_variant_workflow(
        variant=variant,
        zscored_df=zscored_df,
        obs_df=obs_df,
        lineage_to_type=lineage_to_type,
        device=device,
    )

list(variant_results)

Terminal lineage count: 1092
Timepoint matrix shape: (150675, 266)
Z-scored matrix shape: (150675, 266)

[exclude_programmed_death] samples=61815, features=266, groups=496, classes=17
cell_type
neuron        22953
muscle        16041
hypoderm       8935
sheath         2946
socket         2339
epithelium     1704
other          1639
marginal       1310
valve          1125
intestine       626
Name: count, dtype: int64


Phase-2 CV configs:   4%|▍         | 1/24 [1:25:40<32:50:42, 5140.96s/it]


NotImplementedError: The operator 'aten::_cdist_backward' is not currently implemented for the MPS device. If you want this op to be considered for addition please comment on https://github.com/pytorch/pytorch/issues/141287 and mention use-case, that resulted in missing op as well as commit hash 70d99e998b4955e0049d13a98d77ae1b14db1f45. As a temporary fix, you can set the environment variable `PYTORCH_ENABLE_MPS_FALLBACK=1` to use the CPU as a fallback for this op. WARNING: this will be slower than running natively on MPS.

## 6. Validate Results with Simple Checks

These checks confirm that grouped folds remain leakage-free and that each variant produced aligned embeddings, metadata, and out-of-fold predictions.

In [ ]:
for variant_name, bundle in variant_results.items():
    dataset = bundle["dataset"]
    embeddings_df = bundle["embeddings_df"]
    metadata_df = bundle["metadata_df"]
    oof_df = bundle["oof_df"]

    assert not embeddings_df.empty, variant_name
    assert embeddings_df.shape[0] == metadata_df.shape[0], variant_name
    assert oof_df.shape[0] == metadata_df.shape[0], variant_name
    assert len(bundle["selected_features"]) == bundle["best_phase2_config"]["n_select"], variant_name

    folds = build_cv_folds(
        dataset["X"],
        dataset["y"],
        n_splits=N_SPLITS,
        seed=SEED,
        groups=dataset["groups"],
    )
    for train_idx, val_idx in folds:
        train_groups = set(dataset["groups"][train_idx])
        val_groups = set(dataset["groups"][val_idx])
        assert train_groups.isdisjoint(val_groups), variant_name

print("All grouped workflow checks passed.")

## 7. Inspect Intermediate Outputs

The final section summarizes each variant, shows the top grouped-CV configurations, prints per-lineage held-out accuracy tables, and renders compact diagnostics for confusion, time trends, feature stability, and the learned embedding geometry.

In [ ]:
overview_rows = []
for variant_name, bundle in variant_results.items():
    overview_rows.append({
        "variant": variant_name,
        "n_samples": bundle["dataset"]["X"].shape[0],
        "n_groups": bundle["dataset"]["n_groups"],
        "n_classes": len(bundle["dataset"]["class_names"]),
        "grouped_oof_accuracy": bundle["oof_df"]["correct"].mean(),
        "final_accuracy": bundle["final_metrics"]["argmax_accuracy"],
        "expected_target_probability": bundle["final_metrics"]["expected_target_probability"],
        "soft_cross_entropy": bundle["final_metrics"]["soft_cross_entropy"],
        "selected_features": len(bundle["selected_features"]),
    })

overview_df = pd.DataFrame(overview_rows).sort_values("grouped_oof_accuracy", ascending=False)
display(overview_df.round(4))

for variant_name, bundle in variant_results.items():
    print(f"\n=== {variant_name} ===")
    print("Saved files:")
    for label, path in bundle["saved_paths"].items():
        print(f"  {label:18s} -> {path.name}")

    print("\nTop grouped Phase-2 configurations:")
    display(bundle["phase2_summary_df"].head(5).round(4))

    per_lineage_accuracy = (
        bundle["oof_df"]
        .groupby("lineage_name")
        .agg(accuracy=("correct", "mean"), n=("correct", "size"))
        .sort_values(["accuracy", "n"], ascending=[False, False])
    )
    print("Best-performing held-out lineages:")
    display(per_lineage_accuracy.head(10).round(4))
    print("Lowest-performing held-out lineages:")
    display(per_lineage_accuracy.tail(10).round(4))

    plot_variant_diagnostics(bundle)